# Step 6 — Train and validate without geographic leakage

This notebook connects the real SWED loader to a complete epoch loop. Run the optional data-download cell in notebook 01 first.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/hriship618/coastline-image-segmentation.git"
repository = Path("/content/coastline-image-segmentation")
if not repository.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repository)], check=True)
else:
    subprocess.run(["git", "-C", str(repository), "pull", "--ff-only"], check=True)
os.chdir(repository)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml]"], check=True)
source_directory = str(repository / "src")
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)
print(f"Working from: {repository}")

## 1. Discover examples and split whole regions
Each Sentinel filename contains an MGRS tile ID such as `T48QYJ`. Every image from one tile is assigned to exactly one split. This prevents neighboring views of the same coastline from appearing in both training and evaluation.

In [ ]:
from pathlib import Path
from coastlearn.data import discover_swed_pairs, split_pairs_by_region, swed_region_id

# Notebook 01 extracts the complete dataset here when DATASET_VARIANT = "full".
data_root = Path("/content/data/swed_full")
if not data_root.exists():
    raise FileNotFoundError(
        f"{data_root} was not found. In notebook 01, set DOWNLOAD_DATA = True and DATASET_VARIANT = 'full', then run its download cell."
    )
pairs = discover_swed_pairs(data_root)
splits = split_pairs_by_region(pairs, validation_fraction=0.2, test_fraction=0.2, seed=7)
for name, split in (("train", splits.train), ("validation", splits.validation), ("test", splits.test)):
    regions = sorted({swed_region_id(pair) for pair in split})
    print(name, "examples=", len(split), "regions=", regions)

## 2. Build DataLoaders
A DataLoader groups examples into batches. Only training is shuffled; validation and test ordering remains stable.

In [ ]:
from coastlearn.data import FIVE_BANDS, build_dataloaders

loaders = build_dataloaders(splits, bands=FIVE_BANDS, batch_size=8, num_workers=2)
batch = next(iter(loaders["train"]))
print("images:", batch["image"].shape)
print("masks: ", batch["mask"].shape)

## 3. Choose one model
Start with ResNet34-UNet. Keeping model selection to one line lets the same training engine later run ConvNeXt and DINOv3.

In [ ]:
import torch
from coastlearn.models import build_resnet34_unet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_resnet34_unet(in_channels=5, num_classes=2, pretrained=True).to(device)
print("device:", device)

## 4. Train and validate
One epoch is one complete pass through the training loader. Validation runs afterward with gradients disabled. The checkpoint is replaced only when validation mean IoU improves. Use three epochs first to verify the pipeline, then increase the value.

In [ ]:
from coastlearn.training import (
    build_cross_entropy_loss,
    build_finetuning_optimizer,
    fit,
)

loss_function = build_cross_entropy_loss(ignore_index=255).to(device)
optimizer = build_finetuning_optimizer(
    model, backbone_learning_rate=1e-5, head_learning_rate=1e-3
)
history = fit(
    model=model,
    train_loader=loaders["train"],
    validation_loader=loaders["validation"],
    optimizer=optimizer,
    loss_function=loss_function,
    device=device,
    epochs=3,
    checkpoint_path="/content/checkpoints/resnet34_unet_best.pt",
    patience=3,
)

## 5. Understand IoU
For one class, intersection is the number of pixels correctly predicted as that class. Union is every pixel predicted or labeled as that class. `IoU = intersection / union`. Mean IoU averages the land and water IoUs.

In [ ]:
print("last epoch:", history[-1])
print("best checkpoint: /content/checkpoints/resnet34_unet_best.pt")

## Why the test split remains untouched
Validation guides model and training choices. Test data is reserved until those choices are finished. Repeatedly checking test performance would indirectly tune the project to the test regions and weaken the generalization claim.